In [ ]:
# Optional for Colab: uncomment only when storing files in your own Drive.
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os

# Portable local Colab workspace; override via SJK_PORTFOLIO_DIR if desired.
PROJECT_DIR = Path(os.environ.get('SJK_PORTFOLIO_DIR', '/content/sjk_portfolio'))
DOWNLOAD_DIR = PROJECT_DIR / 'audio'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
path = str(PROJECT_DIR / 'metadata')


In [ ]:
!pip install pytube pandas
# ffmpeg must be available for the final conversion cell.


Import modules to read and download from youtube

In [ ]:
import re
import csv
import os
import pandas as pd
import subprocess
import os
import subprocess
from pytube import Playlist, YouTube

Prompt for video or playlist url and a directory to save to. Checks to see if a metadata.csv file exists and if not, creates one with the headings "Title, Video ID, Channel ID, Description, Length (sec), Date Uploaded"

In [ ]:
YOUTUBE_STREAM_AUDIO = '140'
url = input("Enter the url of the playlist or video to download: ")
DOWNLOAD_DIR = str(DOWNLOAD_DIR)
#input("Enter the directory you would like to save to: ")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
file_path = os.path.join(DOWNLOAD_DIR, 'metadata.csv')
if not os.path.exists(file_path):
    with open(file_path, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Title', 'Video ID', 'Channel ID', 'Description', 'Length (sec)', 'Date Uploaded'])

data = []
existing_data = pd.read_csv(file_path)

Check if url is a playlist or single video. Take playlist and iterate through each video first checking directory. If the video's data exists in csv file then checking if audio file exists. Append missing data to end of csv if missing and download audio if missing otherwise skip.

If not in a playlist take video url and check directory. If video's data exists in csv file and if the audio file exists.

In [ ]:
if "list" in url:
    pl = Playlist(url)
    pl._video_regex = re.compile(r"\"url\":\"(/watch\?v=[\w-]*)")
    if pl:
        for video in pl.videos:
            yt = video
            if yt.video_id in existing_data['Video ID'].values:
                print(f"{yt.title} already exists in the file.Checking for audio.")
                audio_file_path = os.path.join(DOWNLOAD_DIR, f"{yt.video_id}.mp4")
                if os.path.exists(audio_file_path):
                    print(f"Audio for video: {yt.title} already exists.")
                else:
                    print(f"Audio for video: {yt.title} not found. Downloading")
                    yt.streams.get_by_itag(YOUTUBE_STREAM_AUDIO).download(output_path=DOWNLOAD_DIR, filename=f"{yt.video_id}.mp4")
            else:
                print(f"New video: {yt.title} Saving metadata and downloading audio")
                data.append([yt.title, yt.video_id, yt.channel_id, yt.description, yt.length, yt.publish_date])
                yt.streams.get_by_itag(YOUTUBE_STREAM_AUDIO).download(output_path=DOWNLOAD_DIR, filename=f"{yt.video_id}.mp4")
    else:
        yt = YouTube(url)
        audio_file_path = os.path.join(DOWNLOAD_DIR, f"{yt.video_id}.mp4")
        if yt.video_id in existing_data['Video ID'].values:
            print(f"{yt.title} already exists in the file.Checking for audio.")
            audio_file_path = os.path.join(DOWNLOAD_DIR, f"{yt.video_id}.mp4")
            if os.path.exists(audio_file_path):
                print(f"Audio for video: {yt.title} already exists.")
            else:
                print(f"Audio for video: {yt.title} not found. Downloading")
                yt.streams.filter(type='audio', subtype='mp4').first().download(output_path=DOWNLOAD_DIR, filename=f"{yt.video_id}.mp4")
        else:
            data.append([yt.title, yt.video_id, yt.channel_id, yt.description, yt.length, yt.publish_date])
            yt.streams.filter(type='audio', subtype='mp4').first().download(output_path=DOWNLOAD_DIR, filename=f"{yt.video_id}.mp4")

else:
    yt = YouTube(url)
    if yt.video_id in existing_data['Video ID'].values:
        print(f"{yt.video_id} already exists in the file.Checking for audio.")
        audio_file_path = os.path.join(DOWNLOAD_DIR, f"{yt.video_id}.mp4")
        if os.path.exists(audio_file_path):
            print(f"Audio for video: {yt.video_id} already exists.")
        else:
            print(f"Audio for video: {yt.video_id} not found. Downloading")
            yt.streams.filter(type='audio', subtype='mp4').first().download(output_path=DOWNLOAD_DIR, filename=f"{yt.video_id}.mp4")
    else:
        data.append([yt.title, yt.video_id, yt.channel_id, yt.description, yt.length, yt.publish_date])
        yt.streams.filter(type='audio', subtype='mp4').first().download(output_path=DOWNLOAD_DIR, filename=f"{yt.video_id}.mp4")

Save the csv file.

In [ ]:
df = pd.DataFrame(data,columns=['Title', 'Video ID', 'Channel ID', 'Description', 'Length (sec)', 'Date Uploaded'])
df.to_csv(file_path, mode='a', index=False, header=False)

print("The videos have been downloaded!")

In [ ]:
# Optional alternative run from the original notebook. Do not run this cell after cells 6–10 unless you intend to repeat the download workflow.
YOUTUBE_STREAM_AUDIO = '140'
url = input("Enter the url of the playlist or video to download: ")
DOWNLOAD_DIR = str(DOWNLOAD_DIR) #input("Enter the directory you would like to save to: ")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
file_path = os.path.join(DOWNLOAD_DIR, 'metadata.csv')
if not os.path.exists(file_path):
    with open(file_path, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Title', 'Video ID', 'Channel ID', 'Description', 'Length (sec)', 'Date Uploaded'])

data = []
existing_data = pd.read_csv(file_path)

if "list" in url:
    pl = Playlist(url)
    pl._video_regex = re.compile(r"\"url\":\"(/watch\?v=[\w-]*)")
    if pl:
        for video in pl.videos:
            yt = video
            if yt.video_id in existing_data['Video ID'].values:
                print(f"{yt.title} already exists in the file.Checking for audio.")
                audio_file_path = os.path.join(DOWNLOAD_DIR, f"{yt.video_id}.mp4")
                if os.path.exists(audio_file_path):
                    print(f"Audio for video: {yt.title} already exists.")
                else:
                    print(f"Audio for video: {yt.title} not found. Downloading")
                    yt.streams.get_by_itag(YOUTUBE_STREAM_AUDIO).download(output_path=DOWNLOAD_DIR, filename=f"{yt.video_id}.mp4")
            else:
                print(f"New video: {yt.title} Saving metadata and downloading audio")
                data.append([yt.title, yt.video_id, yt.channel_id, yt.description, yt.length, yt.publish_date])
                yt.streams.get_by_itag(YOUTUBE_STREAM_AUDIO).download(output_path=DOWNLOAD_DIR, filename=f"{yt.video_id}.mp4")
    else:
        yt = YouTube(url)
        audio_file_path = os.path.join(DOWNLOAD_DIR, f"{yt.video_id}.mp4")
        if yt.video_id in existing_data['Video ID'].values:
            print(f"{yt.title} already exists in the file.Checking for audio.")
            audio_file_path = os.path.join(DOWNLOAD_DIR, f"{yt.video_id}.mp4")
            if os.path.exists(audio_file_path):
                print(f"Audio for video: {yt.title} already exists.")
            else:
                print(f"Audio for video: {yt.title} not found. Downloading")
                yt.streams.filter(type='audio', subtype='mp4').first().download(output_path=DOWNLOAD_DIR, filename=f"{yt.video_id}.mp4")
        else:
            data.append([yt.title, yt.video_id, yt.channel_id, yt.description, yt.length, yt.publish_date])
            yt.streams.filter(type='audio', subtype='mp4').first().download(output_path=DOWNLOAD_DIR, filename=f"{yt.video_id}.mp4")

else:
    yt = YouTube(url)
    if yt.video_id in existing_data['Video ID'].values:
        print(f"{yt.video_id} already exists in the file.Checking for audio.")
        audio_file_path = os.path.join(DOWNLOAD_DIR, f"{yt.video_id}.mp4")
        if os.path.exists(audio_file_path):
            print(f"Audio for video: {yt.video_id} already exists.")
        else:
            print(f"Audio for video: {yt.video_id} not found. Downloading")
            yt.streams.filter(type='audio', subtype='mp4').first().download(output_path=DOWNLOAD_DIR, filename=f"{yt.video_id}.mp4")
    else:
        data.append([yt.title, yt.video_id, yt.channel_id, yt.description, yt.length, yt.publish_date])
        yt.streams.filter(type='audio', subtype='mp4').first().download(output_path=DOWNLOAD_DIR, filename=f"{yt.video_id}.mp4")

df = pd.DataFrame(data,columns=['Title', 'Video ID', 'Channel ID', 'Description', 'Length (sec)', 'Date Uploaded'])
df.to_csv(file_path, mode='a', index=False, header=False)

print("The videos have been downloaded!")

Convert the downloaded .mp4 files into .mp3 files and save them to a new folder to be read by OpenAI whisper.

In [ ]:
folder = DOWNLOAD_DIR
mp3_folder = os.path.join(folder, "mp3_converted")

if not os.path.exists(mp3_folder):
    os.makedirs(mp3_folder)

for filename in os.listdir(DOWNLOAD_DIR):
    if filename.endswith(".mp4"):
        mp4_file = os.path.join(DOWNLOAD_DIR, filename)
        mp3_file = os.path.join(mp3_folder, filename[:-4] + ".mp3")
        subprocess.call(['ffmpeg', '-i', mp4_file, mp3_file])